# 04. Master manifest 만들기

검증한 FMA REAL 296곡과 Echoes TTA FAKE 3,162곡을 하나의 표로 합친다. 이 표의 `original_audio`는 다음 단계에서 원곡 단위로 분할하는 기준이다.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_MAPPING = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
FMA_AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"

OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("ECHOES_MANIFEST  :", ECHOES_MANIFEST)
print("FMA_MAPPING      :", FMA_MAPPING)
print("OUTPUT_PATH      :", OUTPUT_PATH)

PROJECT_ROOT     : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project
ECHOES_MANIFEST  : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/raw/Echoes/Echoes/dataset_manifest.csv
FMA_MAPPING      : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/fma_real_mapping.csv
OUTPUT_PATH      : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest.csv


## 1. Echoes manifest 로드 및 Clean TTA 재구성

Echoes manifest에서 `type == 'TTA'`인 행만 사용한다.

이전 품질 검증에서 동일한 physical path가 서로 다른 `original_audio`에 연결된 중복 TTA 파일이 확인되었다.
따라서 `path_in_dataset`이 중복된 경우 해당 중복 행을 모두 제외하여 Clean TTA를 구성한다.

In [3]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("Echoes manifest rows:", len(echoes))
print("Columns:", echoes.columns.tolist())

tta = echoes[echoes["type"] == "TTA"].copy()

print("\nTTA rows:", len(tta))
print("TTA unique original_audio:", tta["original_audio"].nunique())

Echoes manifest rows: 4468
Columns: ['path_in_dataset', 'original_audio', 'generator', 'type', 'genre', 'description', 'duration']

TTA rows: 3165
TTA unique original_audio: 296


In [4]:
# Echoes manifest 로드 및 Clean TTA 재구성
dup_path_mask = tta["path_in_dataset"].duplicated(keep=False)
duplicated_tta = tta[dup_path_mask].copy()

print("Duplicated TTA rows        :", len(duplicated_tta))
print("Duplicated TTA unique paths:", duplicated_tta["path_in_dataset"].nunique())

if len(duplicated_tta):
    display(
        duplicated_tta[
            ["path_in_dataset", "original_audio", "generator", "genre"]
        ].sort_values("path_in_dataset")
    )

Duplicated TTA rows        : 3
Duplicated TTA unique paths: 1


,path_in_dataset,original_audio,generator,genre
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,Rock


In [5]:
# Echoes manifest 로드 및 Clean TTA 재구성
tta_clean = tta[~dup_path_mask].copy().reset_index(drop=True)

print("===== CLEAN TTA =====")
print("Rows                 :", len(tta_clean))
print("Unique original_audio:", tta_clean["original_audio"].nunique())
print("Generators           :", tta_clean["generator"].nunique())

print("\nGenre distribution:")
print(tta_clean["genre"].value_counts())

===== CLEAN TTA =====
Rows                 : 3162
Unique original_audio: 296
Generators           : 12

Genre distribution:
genre
Electronic    1131
Rock          1127
Pop            904
Name: count, dtype: int64


### Clean TTA 기대값

- Rows: **3,162**
- Unique `original_audio`: **296**
- Generators: **12**

## 2. FMA REAL 매핑 로드

QC가 완료된 최종 FMA REAL mapping을 불러온다.

각 `original_audio`에는 하나의 REAL track만 존재해야 한다.

In [6]:
real_mapping = pd.read_csv(FMA_MAPPING)
real_mapping["track_id"] = real_mapping["track_id"].astype(int)

print("===== FMA REAL =====")
print("Rows                 :", len(real_mapping))
print("Unique original_audio:", real_mapping["original_audio"].nunique())
print("Unique track_id      :", real_mapping["track_id"].nunique())

display(real_mapping.head())

===== FMA REAL =====
Rows                 : 296
Unique original_audio: 296
Unique track_id      : 296


,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True


## 3. REAL과 FAKE의 original_audio 집합 비교

REAL과 FAKE가 동일한 296개의 source family를 공유하는지 확인한다.

이 검사는 이후 `original_audio` 단위 split을 수행하기 위한 핵심 조건이다.

In [7]:
real_groups = set(real_mapping["original_audio"])
fake_groups = set(tta_clean["original_audio"])

only_real = sorted(real_groups - fake_groups)
only_fake = sorted(fake_groups - real_groups)

print("REAL groups:", len(real_groups))
print("FAKE groups:", len(fake_groups))
print("Only REAL  :", len(only_real))
print("Only FAKE  :", len(only_fake))

if only_real:
    print("\nOnly REAL examples:", only_real[:10])

if only_fake:
    print("\nOnly FAKE examples:", only_fake[:10])

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert (
    real_groups == fake_groups
), "REAL과 FAKE의 original_audio 집합이 일치하지 않습니다."

print("\noriginal_audio group match: PASS")

REAL groups: 296
FAKE groups: 296
Only REAL  : 0
Only FAKE  : 0

original_audio group match: PASS


## 4. REAL manifest 생성

REAL 데이터는 FMA의 검증 완료된 30초 clip을 사용한다.

- `label = REAL`
- `label_id = 0`
- `source = FMA`
- `generator = real`

In [15]:
# REAL manifest 생성
real_manifest = real_mapping.copy()

real_manifest["label"] = "REAL"
real_manifest["label_id"] = 0
real_manifest["source"] = "FMA"
real_manifest["generator"] = pd.NA
real_manifest["description"] = ""

real_manifest["audio_path"] = real_manifest["track_id"].apply(
    lambda tid: str(
        Path("data/raw/FMA/selected_30s")
        / f"{int(tid) // 1000:03d}"
        / f"{int(tid):06d}.mp3"
    )
)

real_manifest["path_in_dataset"] = pd.NA

display(
    real_manifest[
        ["original_audio", "label", "genre", "generator", "track_id", "audio_path"]
    ].head()
)

,original_audio,label,genre,generator,track_id,audio_path
0,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,Electronic,<NA>,140932,data/raw/FMA/selected_30s/140/140932.mp3
1,1984 - Punk Rock Opera,REAL,Rock,<NA>,149410,data/raw/FMA/selected_30s/149/149410.mp3
2,2 (Wasn't There) - Isle of Pine,REAL,Rock,<NA>,66449,data/raw/FMA/selected_30s/066/066449.mp3
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,Electronic,<NA>,114244,data/raw/FMA/selected_30s/114/114244.mp3
4,3 am West End - statusq,REAL,Electronic,<NA>,112378,data/raw/FMA/selected_30s/112/112378.mp3


## 5. FAKE manifest 생성

Clean TTA의 모든 AI 생성 음악을 FAKE 데이터로 구성한다.

- `label = FAKE`
- `label_id = 1`
- `source = Echoes`
- `generator`는 실제 생성기 이름 유지

In [16]:
# FAKE manifest 생성
fake_manifest = tta_clean.copy()

fake_manifest["label"] = "FAKE"
fake_manifest["label_id"] = 1
fake_manifest["source"] = "Echoes"
fake_manifest["track_id"] = pd.NA

fake_manifest["audio_path"] = fake_manifest["path_in_dataset"].apply(
    lambda p: str(Path("data/raw/Echoes/Echoes") / str(p))
)

display(
    fake_manifest[
        ["original_audio", "label", "genre", "generator", "description", "audio_path"]
    ].head()
)

,original_audio,label,genre,generator,description,audio_path
0,"10,000 People Chanting, ""I'm an Individual"" - ...",FAKE,Electronic,acestep,"cinematic, idm, downtempo, layered, swelling, ...",data/raw/Echoes/Echoes/TTA/acestep/10000_Peopl...
1,1984 - Punk Rock Opera,FAKE,Rock,acestep,"hardcore-punk, political, d-beat, shouted-chor...",data/raw/Echoes/Echoes/TTA/acestep/1984_Punk_R...
2,2Much (Andy Spinelli & Alex Sánchez House Edit...,FAKE,Electronic,acestep,"house, four-on-the-floor, piano-stabs, filtere...",data/raw/Echoes/Echoes/TTA/acestep/2Much_Andy_...
3,2 (Wasn't There) - Isle of Pine,FAKE,Rock,acestep,"post-rock, melancholic, clean-guitars, spaciou...",data/raw/Echoes/Echoes/TTA/acestep/2_Wasnt_The...
4,3 am West End - statusq,FAKE,Electronic,acestep,"chillhop, nocturnal, lofi, warm, headnod, mell...",data/raw/Echoes/Echoes/TTA/acestep/3_am_West_E...


## 6. REAL + FAKE 통합

두 데이터셋을 공통 schema로 맞춘 뒤 하나의 master manifest로 결합한다.

In [17]:
# REAL + FAKE 통합
master_columns = [
    "original_audio",
    "label",
    "label_id",
    "source",
    "genre",
    "generator",
    "audio_path",
    "track_id",
    "description",
    "path_in_dataset",
]

real_part = real_manifest[master_columns].copy()
fake_part = fake_manifest[master_columns].copy()

master_manifest = pd.concat(
    [real_part, fake_part],
    ignore_index=True,
)

master_manifest.insert(
    0,
    "sample_id",
    [f"sample_{i:05d}" for i in range(len(master_manifest))],
)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master_manifest))
print("Unique original_audio:", master_manifest["original_audio"].nunique())

print("\nLabel distribution:")
print(master_manifest["label"].value_counts())

display(master_manifest.head())

===== MASTER MANIFEST =====
Rows                 : 3458
Unique original_audio: 296

Label distribution:
label
FAKE    3162
REAL     296
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/140/140932.mp3,140932,,<NA>
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,<NA>,data/raw/FMA/selected_30s/149/149410.mp3,149410,,<NA>
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,<NA>,data/raw/FMA/selected_30s/066/066449.mp3,66449,,<NA>
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/114/114244.mp3,114244,,<NA>
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/112/112378.mp3,112378,,<NA>


### 기대값

```text
Rows                  : 3458
Unique original_audio : 296

FAKE : 3162
REAL : 296
```

## 7. 원곡별 REAL/FAKE 구조 검증

각 `original_audio` group에 REAL이 정확히 1개씩 존재하는지 확인한다.
FAKE는 generator 구성에 따라 원곡별 개수가 다를 수 있다.

In [18]:
# 원곡별 REAL/FAKE 구조 검증
group_check = (
    master_manifest.groupby("original_audio")
    .agg(
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
        genre_count=("genre", "nunique"),
        generator_count=("generator", "nunique"),
    )
    .reset_index()
)

bad_real_count = group_check[group_check["real_count"] != 1]
bad_genre_count = group_check[group_check["genre_count"] != 1]

print("Groups                :", len(group_check))
print("REAL count != 1       :", len(bad_real_count))
print("Genre count != 1      :", len(bad_genre_count))

display(group_check.head())

Groups                : 296
REAL count != 1       : 0
Genre count != 1      : 0


,original_audio,total_samples,real_count,fake_count,genre_count,generator_count
0,"10,000 People Chanting, ""I'm an Individual"" - ...",6,1,5,1,5
1,1984 - Punk Rock Opera,14,1,13,1,12
2,2 (Wasn't There) - Isle of Pine,17,1,16,1,12
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,6,1,5,1,5
4,3 am West End - statusq,6,1,5,1,5


## 8. 실제 오디오 파일 존재 여부 확인

`master_manifest.csv`의 모든 `audio_path`가 실제 파일을 가리키는지 확인한다.

In [19]:
# 실제 오디오 파일 존재 여부 확인
master_manifest["file_exists"] = master_manifest["audio_path"].apply(
    lambda p: (PROJECT_ROOT / p).exists()
)

missing_files = master_manifest[~master_manifest["file_exists"]].copy()

print("Total rows   :", len(master_manifest))
print("Files exist :", int(master_manifest["file_exists"].sum()))
print("Missing     :", len(missing_files))

if len(missing_files):
    display(
        missing_files[["sample_id", "original_audio", "label", "audio_path"]].head(30)
    )

Total rows   : 3458
Files exist : 3458
Missing     : 0


## 9. 최종 QC

master manifest가 다음 핵심 조건을 모두 만족하는지 확인한다.

In [20]:
# 최종 QC
qc_summary = pd.DataFrame(
    {
        "check": [
            "master_rows",
            "unique_original_audio",
            "real_rows",
            "fake_rows",
            "groups_with_real_count_not_1",
            "groups_with_genre_count_not_1",
            "missing_audio_files",
            "duplicate_sample_id",
        ],
        "value": [
            len(master_manifest),
            master_manifest["original_audio"].nunique(),
            int((master_manifest["label"] == "REAL").sum()),
            int((master_manifest["label"] == "FAKE").sum()),
            len(bad_real_count),
            len(bad_genre_count),
            len(missing_files),
            int(master_manifest["sample_id"].duplicated().sum()),
        ],
    }
)

display(qc_summary)

core_qc_pass = (
    len(master_manifest) == 3458
    and master_manifest["original_audio"].nunique() == 296
    and int((master_manifest["label"] == "REAL").sum()) == 296
    and int((master_manifest["label"] == "FAKE").sum()) == 3162
    and len(bad_real_count) == 0
    and len(bad_genre_count) == 0
    and len(missing_files) == 0
    and int(master_manifest["sample_id"].duplicated().sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)

,check,value
0,master_rows,3458
1,unique_original_audio,296
2,real_rows,296
3,fake_rows,3162
4,groups_with_real_count_not_1,0
5,groups_with_genre_count_not_1,0
6,missing_audio_files,0
7,duplicate_sample_id,0


===== FINAL RESULT =====
Core QC PASS: True


## 10. master_manifest.csv 저장

최종 QC가 통과한 manifest를 저장한다.

In [21]:
if not core_qc_pass:
    raise RuntimeError("Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

master_manifest.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)
print("Rows :", len(master_manifest))

Saved: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest.csv
Rows : 3458


## 이어지는 기록

이 표의 `original_audio`를 기준으로 05번에서 Train·Validation·Test를 나눈다.

## 합친 곡 목록

- REAL 296곡과 Clean TTA FAKE 3,162곡을 결합해 **3,458행**의 master manifest를 생성했다.
- 고유 `original_audio` 그룹은 296개이며, 각 그룹은 REAL 1개와 일관된 장르를 가진다.
- 연결된 실제 파일은 **3,458/3,458 존재**하며 누락은 0개다.
- 결과를 `data/metadata/master_manifest.csv`에 저장했다.